# Solutions — Set Operations

**Dataset:** `samples.bakehouse.sales_customers`, `samples.wanderbricks.users`

**Topics:** union, unionByName, intersect, subtract, distinct, dropDuplicates

> Reference solutions for practice notebooks.
> **Try the problem yourself first** — only open this when you've made a genuine attempt.
>
> Each solution includes a `# Why:` comment explaining the approach chosen.

In [ ]:
from pyspark.sql import functions as F, types as T
customers = spark.table("samples.bakehouse.sales_customers")
users     = spark.table("samples.wanderbricks.users")

---
## Problem 1 — Union Two DataFrames

In [ ]:
# Why: unionByName matches by column name — safer than union() which matches by position.
# F.lit() creates a constant column with the same value in every row.
bakehouse_countries    = customers.select("country").distinct().withColumn("source", F.lit("bakehouse"))
wanderbricks_countries = users.select("country").distinct().withColumn("source", F.lit("wanderbricks"))

result_1 = bakehouse_countries.unionByName(wanderbricks_countries)

---
## Problem 2 — Find Common Countries

In [ ]:
# Why: intersect() finds rows in both DataFrames — Spark handles the deduplication.
result_2 = customers.select("country").intersect(users.select("country")).orderBy("country")

---
## Problem 3 — Find Countries Unique to Bakehouse

In [ ]:
# Why: subtract() is SQL EXCEPT — keeps rows in left that don't appear in right.
result_3 = customers.select("country").subtract(users.select("country")).orderBy("country")

---
## Problem 4 — Remove Duplicates

In [ ]:
# Why: dropDuplicates on a subset treats those columns as the dedup key.
before = customers.count()
after  = customers.dropDuplicates(["first_name", "last_name", "country"]).count()

result_4 = spark.createDataFrame(
    [("before dedup", before), ("after dedup", after)],
    ["step", "row_count"]
)

---
## Problem 5 — Union All Three User Sources

In [ ]:
# Why: align schemas with select + alias before unionByName so column names match exactly.
from_customers = customers.select(
    F.col("customerID").alias("user_id"),
    F.col("first_name").alias("name"),
    F.col("country"),
    F.lit("bakehouse").alias("source")
)

from_users = users.select(
    F.col("user_id"),
    F.col("name"),
    F.col("country"),
    F.lit("wanderbricks").alias("source")
)

result_5 = (
    from_customers.unionByName(from_users)
                  .groupBy("source")
                  .count()
                  .withColumnRenamed("count", "user_count")
)